# Modul KKA: Informed Search

## Tujuan praktikum

Pada praktikum ini, Anda akan merumuskan masalah pencarian, menelusuri urutan
kunjungan simpul, dan menggunakan heuristik, serta membandingkan biaya dan jumlah ekspansi beberapa algoritma.

### Formulasi masalah

Contoh labirin menggunakan aturan berikut.

| Komponen | Definisi |
|---|---|
| Keadaan (*state*) | Koordinat `(baris, kolom)` pada sel yang dapat dilalui |
| Keadaan awal | `(0, 0)` |
| Aksi | Bergerak satu sel ke kanan, bawah, kiri, atau atas |
| Uji tujuan (*goal test*) | Koordinat sama dengan `(4, 4)` |
| Biaya lintasan $g(n)$ | Jumlah perpindahan dari keadaan awal; setiap perpindahan berbiaya 1 |

Gerakan tidak boleh keluar dari labirin atau menembus dinding. Satu keadaan dapat
dicapai melalui beberapa lintasan. Simpul pencarian menyimpan keadaan beserta
informasi lintasannya, seperti induk, kedalaman, dan biaya.

### Kriteria perbandingan

- **Kelengkapan (*complete*):** apakah algoritma menemukan solusi jika solusi ada?
- **Optimalitas:** apakah solusi yang ditemukan memiliki biaya minimum?
- **Waktu:** berapa banyak simpul yang diekspansi dan berapa lama pencarian berlangsung?
- **Memori:** berapa banyak informasi yang harus disimpan selama pencarian?

Dalam analisis kompleksitas, $b$ adalah jumlah cabang maksimum per simpul,
$d$ kedalaman solusi terdangkal, $m$ kedalaman maksimum pohon pencarian, dan
$\ell$ batas kedalaman pada DLS. Panjang lintasan dihitung dalam perpindahan:
`len(path) - 1`.

## Persiapan praktikum

Lengkapi identitas berikut. Petunjuk menjalankan Jupyter dan Google Colab tersedia di README.

| Nama | NRP |
|---|---|
| | |

### Mengenali bagian yang harus diisi

Setiap bagian tugas diberi komentar **`# TODO [kode]`**, diikuti penanda
**`________("kode")`** pada posisi jawaban. Nomor pada komentar dan penanda
sama, misalnya U01.

- Ganti **seluruh pemanggilan** `________("U01")`, termasuk tanda kurung dan
  kodenya, dengan ekspresi atau perintah Python yang diminta pada komentar.
- Untuk tugas satu fungsi penuh, tulis implementasi di antara komentar
  `# __ MULAI JAWABAN __` dan `# __ AKHIR JAWABAN __`. Hapus baris penandanya.
- Kode di luar penanda sudah disediakan. Sel bertanda **PENGUJIAN** tidak perlu
  diubah; jalankan setelah fungsi latihan selesai diisi.

Contoh: jika diminta menghitung dua kali x, ubah `nilai = ________("X01")`
menjadi `nilai = 2 * x`.

Jalankan sel berurutan. Setelah mengubah fungsi, jalankan kembali sel definisi
dan sel pengujiannya. Pesan **BELUM DIISI** menyebut kode TODO yang masih kosong.
Jika `assert` gagal, periksa kembali jawaban Anda. Fungsi penanda di bawah
digunakan untuk menampilkan pesan tersebut dan tidak perlu diubah.

Seluruh impor ada pada sel persiapan. Setiap algoritma didefinisikan langsung pada bagian materinya, diikuti contoh pemanggilan dan latihan.

In [ ]:
from pathlib import Path
from collections import deque
from itertools import count
from time import perf_counter
import heapq
import random
import sys
import math

import numpy as np

# Jalankan sel persiapan sebelum sel materi atau latihan.
module_root = next((p for p in (Path.cwd(), Path.cwd().parent)
                    if (p / 'data/search_lab.py').is_file()), None)
if module_root is None:
    raise FileNotFoundError('Folder data/ tidak ditemukan. Ikuti petunjuk pada README.')
sys.path.insert(0, str(module_root.resolve()))

from data.search_lab import MAZE, START, GOAL, DIRECTIONS, animate_expansion
from data.search_lab import PUZZLE_START, PUZZLE_GOAL

random.seed(42)
np.random.seed(42)

In [ ]:
# FUNGSI PENANDA LATIHAN — tidak perlu diubah.
def ________(kode):
    raise NotImplementedError(f"TODO [{kode}] belum diisi. Lihat komentar TODO dengan kode yang sama.")

## 1. Greedy Best-First Search (GBFS)

![](data/GBFS.png)

GBFS memilih simpul dengan perkiraan biaya tersisa $h(n)$ paling kecil.
Berbeda dengan BFS yang memakai kedalaman, GBFS memakai heuristik untuk
menentukan urutan pencarian. Biaya yang sudah ditempuh, $g(n)$, tidak ikut
menentukan prioritasnya.

### Fungsi heuristik

Heuristik $h(n)$ memperkirakan biaya dari simpul n ke tujuan. Pada GBFS,
prioritasnya adalah $f(n)=h(n)$. Estimasi yang kecil membuat suatu simpul
didahulukan, tetapi tidak membuktikan bahwa lintasan melalui simpul itu paling murah.

### Antrean prioritas dengan `heapq`

`heapq.heappush` memasukkan entri dan `heapq.heappop` mengeluarkan entri dengan
prioritas terkecil. Kedua operasi membutuhkan waktu $O(\log n)$ untuk n entri.
Prioritas GBFS adalah $h$, UCS adalah $g$, dan A* adalah $g+h$.

Jika dua entri memiliki prioritas sama, urutannya ditentukan oleh komponen tuple
berikutnya. Eksperimen perbandingan menggunakan pencacah agar entri yang lebih
dulu masuk diproses lebih dulu.

### Cara kerja

1. Masukkan simpul awal dengan prioritas $h(start)$.
2. Keluarkan simpul dengan heuristik terkecil dan periksa tujuan.
3. Bangkitkan tetangga yang belum dikunjungi, lalu masukkan ke antrean.
4. Ulangi hingga tujuan ditemukan atau antrean kosong.

GBFS dapat mengurangi pencarian jika heuristiknya membantu, tetapi tidak menjamin
solusi optimal. Pada graf berhingga, pencatatan simpul yang sudah dikunjungi
mencegah pencarian berulang tanpa akhir.

![](data/studi_case_GBFS.png)

In [ ]:
graph_gbfs = {
    'A': ['B', 'C', 'D'],
    'B': ['A', 'E'],
    'C': ['A', 'F', 'E'],
    'D': ['A', 'F'],
    'E': ['B', 'C', 'H'],
    'F': ['C', 'D', 'G'],
    'H': ['E', 'G'],
    'G': ['F', 'H']
}

In [ ]:
# heuristik
h_gbfs = {
    'A': 40,
    'B': 32,
    'C': 25,
    'D': 35,
    'E': 19,
    'F': 17,
    'H': 10,
    'G': 0
}

In [ ]:
def gbfs_graph(graph, start, goal, h):
    pq = []
    heapq.heappush(pq, (h[start], start, [start]))

    visited = set()

    while pq:
        h_val, node, path = heapq.heappop(pq)

        if node == goal:
            return path

        if node not in visited:
            visited.add(node)

            for neighbor in graph[node]:
                if neighbor not in visited:
                    heapq.heappush(pq, (h[neighbor], neighbor, path + [neighbor]))

    return None

In [ ]:
print(gbfs_graph(graph_gbfs, 'A', 'G', h_gbfs))

### Latihan dasar: GBFS pada labirin

Lengkapi fungsi heuristik dan `gbfs_grid` untuk mencari lintasan pada `MAZE`
dari `START=(0, 0)` ke `GOAL=(4, 4)`. Sel 0 dapat dilalui dan sel 1 adalah dinding.
Gunakan jarak Manhattan karena gerakan hanya diizinkan dalam empat arah.

Pertanyaan:
1. Mengapa nilai h yang kecil belum tentu menghasilkan total biaya yang kecil?
2. Apakah heuristik admissible cukup untuk menjamin optimalitas GBFS?
3. Bagaimana urutan pemrosesan simpul dengan prioritas sama memengaruhi hasil?

In [ ]:
# BAGIAN TUGAS — isi 1 penanda: I01.
try:
    # lengkapilah kode yang rumpang berikut


    def gbfs_heuristic(a, b):
        # TODO [I01]: Rumus Manhattan grid 4 arah.
        return ________("I01")

    def gbfs_grid(maze, start, goal):
        rows, cols = len(maze), len(maze[0])

        pq = []
        heapq.heappush(pq, (gbfs_heuristic(start, goal), start, [start]))

        visited = set()

        while pq:
            h_val, node, path = heapq.heappop(pq)

            if node == goal:
                return path

            if node not in visited:
                visited.add(node)

                x, y = node
                moves = [(0,1), (1,0), (0,-1), (-1,0)]

                for dx, dy in moves:
                    nx, ny = x + dx, y + dy

                    neighbor = (nx, ny)

                    if 0 <= nx < rows and 0 <= ny < cols:
                        if maze[nx][ny] == 0 and neighbor not in visited:
                            heapq.heappush(
                                pq,
                                (gbfs_heuristic(neighbor, goal), neighbor, path + [neighbor])
                            )

        return None

    # PENGUJIAN — jalankan setelah fungsi di atas diisi; tidak perlu diubah.
    path_gbfs = gbfs_grid(MAZE, START, GOAL)
    assert path_gbfs[0] == START and path_gbfs[-1] == GOAL
    print('LULUS GBFS:', path_gbfs)
except NotImplementedError as error:
    print("BELUM DIISI:", error)

## 2. A*

A* memilih simpul berdasarkan $f(n)=g(n)+h(n)$, dengan $g(n)$ sebagai biaya
yang sudah ditempuh dan $h(n)$ sebagai perkiraan biaya tersisa. Dengan demikian,
A* mempertimbangkan biaya lintasan sekaligus perkiraan jarak ke tujuan.

### Cara kerja

1. Masukkan simpul awal ke antrean prioritas dengan $g(start)=0$.
2. Keluarkan simpul dengan nilai f terkecil. Abaikan entri yang biayanya sudah usang.
3. Jika simpul tersebut adalah tujuan, kembalikan lintasannya.
4. Hitung biaya menuju setiap tetangga. Jika lebih murah dari catatan sebelumnya,
   perbarui biaya dan induknya, lalu masukkan kembali ke antrean.

Pembaruan ini memungkinkan suatu keadaan diproses kembali ketika ditemukan
lintasan yang lebih murah. A* dapat membutuhkan banyak memori untuk menyimpan
antrean, biaya terbaik, dan informasi lintasan.

![](data/A_star_implementasi.png)

### Contoh: lintasan pada graf berbobot

Setiap sisi pada graf menyatakan biaya perjalanan dan setiap simpul memiliki
heuristik menuju J. Gunakan A* untuk mencari lintasan dari A ke J.

In [ ]:
# Representasi graf (format tuple)
graph_astar = {
    'A': [('B', 6), ('F', 3)],
    'B': [('A', 6), ('C', 3), ('D', 2)],
    'C': [('B', 3), ('D', 1), ('E', 5)],
    'D': [('B', 2), ('C', 1), ('E', 8)],
    'E': [('C', 5), ('D', 8), ('I', 5), ('J', 5)],
    'F': [('A', 3), ('G', 1), ('H', 7)],
    'G': [('F', 1), ('I', 3)],
    'H': [('F', 7), ('I', 2)],
    'I': [('G', 3), ('H', 2), ('E', 5), ('J', 3)],
    'J': [('E', 5), ('I', 3)]
}

# Heuristik
h_astar = {
    'A': 10, 'B': 8, 'C': 5, 'D': 7, 'E': 3,
    'F': 6,  'G': 5, 'H': 3, 'I': 1, 'J': 0
}

In [ ]:
def a_star_graph(adj, start, goal, h):
    heap = [(h[start], 0, start, [start])]  # (f, g, node, path)
    g_score = {start: 0}

    while heap:
        f, g, node, path = heapq.heappop(heap)

        if g > g_score.get(node, float('inf')):
            continue

        # print(f"Mengunjungi: {node} | g={g}, f={f}")

        if node == goal:
            return g, path

        for neighbor, weight in adj[node]:
            new_g = g + weight
            if new_g < g_score.get(neighbor, float('inf')):
                g_score[neighbor] = new_g
                heapq.heappush(heap, (new_g + h[neighbor], new_g, neighbor, path + [neighbor]))

    return None

In [ ]:
cost, path = a_star_graph(graph_astar, 'A', 'J', h_astar)
print(f"Lintasan : {' -> '.join(path)}")
print(f"Total biaya     : {cost}")

### Syarat heuristik dan optimalitas A*

Heuristik disebut **admissible** jika $0\le h(n)\le h^*(n)$, dengan $h^*(n)$
sebagai biaya minimum sebenarnya dari n ke tujuan, serta $h(goal)=0$.
Heuristik disebut **consistent** jika untuk setiap sisi berlaku
$h(n)\le c(n,n')+h(n')$.

A* menemukan solusi optimal jika heuristik admissible, jumlah cabang berhingga,
biaya langkah dibatasi di bawah oleh nilai positif, dan tujuan diperiksa saat
dikeluarkan dari antrean. Keadaan harus dapat dibuka kembali jika ditemukan
biaya g yang lebih kecil. Jika keadaan yang sudah ditutup tidak pernah dibuka
kembali, gunakan heuristik consistent.

Alasannya, sebelum tujuan optimal dipilih, masih ada simpul pada lintasan
optimal di antrean dengan $f(n)\le C^*$. A* tidak akan mendahulukan tujuan yang
biayanya lebih besar dari $C^*$.

Heuristik dapat dibuat dengan menyederhanakan aturan masalah. Contohnya,
mengabaikan dinding menghasilkan jarak Manhattan pada grid empat arah dengan
biaya langkah minimum 1. Pada 8-puzzle, jarak Manhattan menghitung perpindahan
setiap ubin ke posisi tujuannya tanpa memperhitungkan ubin yang menghalangi.
Sel kosong tidak dihitung.

**Dominasi heuristik.** Jika dua heuristik admissible memenuhi $h_2(n)\ge h_1(n)$
untuk semua n, maka $h_2$ mendominasi $h_1$. Heuristik yang lebih kuat mengurangi
simpul dengan $f<C^*$ yang harus diekspansi. Jumlah ekspansi pada $f=C^*$ masih
dapat dipengaruhi urutan pemrosesan. Nilai maksimum dua heuristik admissible
tetap admissible; penjumlahannya belum tentu demikian.

![](data/heuristik.jpg)

### Contoh heuristik: nilai nol

Jika $h(n)=0$ untuk semua simpul, prioritas A* menjadi $g(n)$ sehingga
pencariannya sama dengan UCS.

In [ ]:
def heuristic_zero():
    return 0

### Contoh heuristik: jarak Manhattan

Untuk dua titik pada grid, jarak Manhattan adalah jumlah selisih absolut baris
dan kolomnya: $|r_1-r_2|+|c_1-c_2|$.

In [ ]:
def heuristic_manhattan(positions:dict, node, goal):
    x1, y1 = positions[node]
    x2, y2 = positions[goal]
    return abs(x1 - x2) + abs(y1 - y2)

### Contoh heuristik: jarak Euclidean

Jarak Euclidean adalah jarak garis lurus antara dua titik. Heuristik ini menjadi
batas bawah jika biaya perjalanan tidak lebih kecil daripada jarak garis lurusnya.

In [ ]:
def heuristic_euclidean(positions:dict, node, goal):
    x1, y1 = positions[node]
    x2, y2 = positions[goal]
    return math.sqrt((x1 - x2) ** 2 + (y1 - y2) ** 2)

### Latihan dasar: A* pada 8-puzzle

Lengkapi heuristik Manhattan dan fungsi A* untuk mengubah susunan awal menjadi
`PUZZLE_GOAL_GRID` dalam jumlah gerakan minimum. Angka 0 menyatakan sel kosong. Satu gerakan
menukar sel kosong dengan ubin di sebelahnya secara horizontal atau vertikal.

Pertanyaan:
1. Mengapa sel kosong tidak ikut dihitung dalam heuristik?
2. Apa yang berubah jika heuristik selalu bernilai 0?
3. Kapan biaya terbaik suatu keadaan perlu diperbarui?

Fungsi latihan bernama `a_star_puzzle`; contoh graf sebelumnya memakai `a_star_graph`. Keduanya memiliki nama berbeda agar dapat dipanggil kembali setelah seluruh notebook dijalankan.

![](data/PR_A_star.png)

In [ ]:
PUZZLE_START_GRID = ((1,2,3),
         (0,4,6),
         (7,5,8))

PUZZLE_GOAL_GRID = ((1,2,3),
        (4,5,6),
        (7,8,0))

In [ ]:
# BAGIAN TUGAS — isi 2 penanda: I02, I03.
try:
    # gunakan manhattan distance
    def puzzle_heuristic(state):
        # Hitung total jarak manhattan setiap tile ke posisi tujuannya di PUZZLE_GOAL_GRID
        dist = 0
        for i in range(3):
            for j in range(3):
                val = state[i][j]
                if val != 0:
                    # TODO [I02]: Baris goal untuk tile val.
                    goal_x = ________("I02")
                    # TODO [I03]: Kolom goal untuk tile val.
                    goal_y = ________("I03")
                    dist += abs(i-goal_x) + abs(j-goal_y)

        return dist
except NotImplementedError as error:
    print("BELUM DIISI:", error)

In [ ]:
def get_puzzle_neighbors(state):
    neighbors = []

    for i in range(3):
        for j in range(3):
            if state[i][j] == 0:
                x, y = i, j
    moves = [(0,1), (1,0), (0,-1), (-1,0)]
    for dx, dy in moves:
        nx, ny = x + dx, y + dy
        if 0 <= nx < 3 and 0 <= ny < 3:
            new_state = [list(row) for row in state]
            new_state[x][y], new_state[nx][ny] = new_state[nx][ny], new_state[x][y]
            neighbors.append(tuple(tuple(row) for row in new_state))
    return neighbors

In [ ]:
# BAGIAN TUGAS — isi 1 penanda: I04.
try:

    def a_star_puzzle(start):
        heap = [(puzzle_heuristic(start), 0, start, [start])]
        g_score = {start: 0}

        while heap:
            f, g, state, path = heapq.heappop(heap)
            if state == PUZZLE_GOAL_GRID:
                return path
            if g > g_score[state]:
                continue
            for neighbor in get_puzzle_neighbors(state):
                new_g = g + 1
                if new_g < g_score.get(neighbor, float('inf')):
                    g_score[neighbor] = new_g
                    # TODO [I04]: Prioritas A*: g+h.
                    heapq.heappush(heap,(________("I04"), new_g, neighbor, path + [neighbor]))
        return None

    # PENGUJIAN — jalankan setelah fungsi di atas diisi; tidak perlu diubah.
    puzzle_path = a_star_puzzle(PUZZLE_START_GRID)
    assert len(puzzle_path)-1 == 3
    print('LULUS A* puzzle:', len(puzzle_path)-1, 'langkah')
except NotImplementedError as error:
    print("BELUM DIISI:", error)

## 3. Iterative Deepening A* (IDA*)

IDA* menjalankan penelusuran mendalam dengan batas nilai $f(n)=g(n)+h(n)$.
Cabang dihentikan sementara ketika nilai f melampaui batas. Jika tujuan belum
ditemukan, batas berikutnya diambil dari nilai f terkecil yang melampaui batas lama.

IDA* menyimpan lintasan dan simpul saudara pada penelusuran saat ini, sehingga
memori penelusurannya $O(bd)$. Kekurangannya adalah ekspansi yang berulang pada
setiap kenaikan batas. Dengan syarat heuristik dan biaya seperti A*, IDA*
tetap menemukan solusi optimal.

### Contoh: kenaikan batas pencarian

Graf menyimpan pasangan tetangga dan biaya sisi. Contoh ini menggunakan $h=0$
agar perubahan batas pencarian mudah ditelusuri.

In [ ]:
def idastar_search(graph, heuristics, start, goal, stats=None):
    threshold = heuristics[start]
    if stats is not None:
        stats.update(thresholds=[], expanded=0, stack_peak=1, trace=[])
    
    def search(node, g, threshold, path):
        if stats is not None:
            stats['stack_peak'] = max(stats['stack_peak'], len(path) + 1)
        f = g + heuristics[node]
        if f > threshold:
            return f, None
        if node == goal:
            return -1, path + [node]
        if stats is not None:
            stats['expanded'] += 1
            stats['trace'].append(node)
            
        min_threshold = float('inf')
        for neighbor, cost in graph.get(node, []):
            if neighbor not in path:
                temp_curr, result_path = search(neighbor, g + cost, threshold, path + [node])
                if temp_curr == -1:
                    return -1, result_path
                if temp_curr < min_threshold:
                    min_threshold = temp_curr
        return min_threshold, None

    path = []
    while True:
        if stats is not None:
            stats['thresholds'].append(threshold)
        temp, result_path = search(start, 0, threshold, path)
        if temp == -1:
            return result_path, threshold
        if temp == float('inf'):
            return None, float('inf')
        threshold = temp

### Penelusuran IDA* pada contoh

| Batas f | Hasil |
|---|---|
| 0 | S diproses; nilai f terkecil yang melampaui batas adalah 2 |
| 2 | A diproses; batas berikutnya 4 |
| 4 | C diproses; batas berikutnya 5 |
| 5 | B juga dapat diproses; batas berikutnya 6 |
| 6 | Lintasan S–A–C–G ditemukan dengan biaya 6 |

Simpul dekat akar diekspansi ulang pada setiap pengulangan.

In [ ]:
graph_ida = {
    'S': [('A', 2), ('B', 5)],
    'A': [('C', 2), ('D', 4)],
    'B': [('D', 1)],
    'C': [('G', 2)],
    'D': [('G', 3)],
    'G': []
}

heuristics_ida = {
    'S': 0, 'A': 0, 'B': 0, 'C': 0, 'D': 0, 'G': 0
}

ida_stats = {}
ida_path, ida_bound = idastar_search(graph_ida, heuristics_ida, 'S', 'G', stats=ida_stats)
assert ida_stats['thresholds'] == [0, 2, 4, 5, 6]
print('Urutan batas f:', ida_stats['thresholds'])
print('LULUS IDA*:', ida_path, 'batas akhir', ida_bound)

### Latihan dasar: IDA* pada grid

Cari lintasan dari `(0, 0)` ke `(4, 4)` pada `grid_1` menggunakan heuristik
Manhattan. Sel 0 dapat dilalui dan sel 1 adalah dinding. Lengkapi pencarian
rekursif dan pembaruan batas f.

Pertanyaan:
1. Mengapa batas berikutnya dipilih dari nilai f terkecil yang melampaui batas?
2. Mengapa batas tidak selalu cukup dinaikkan satu satuan?
3. Simpul mana yang diproses kembali ketika batas dinaikkan?

In [ ]:
# BAGIAN TUGAS — isi 1 penanda: I05.
try:
    def manhattan(a, b):
        return abs(a[0] - b[0]) + abs(a[1] - b[1])

    def get_neighbors(grid, r, c):
        neighbors = []
        for dr, dc in [(0, 1), (1, 0), (0, -1), (-1, 0)]:
            nr, nc = r + dr, c + dc
            if 0 <= nr < len(grid) and 0 <= nc < len(grid[0]) and grid[nr][nc] == 0:
                neighbors.append((nr, nc))
        return neighbors

    def idastar_grid(grid, start, goal):
        def search(node, g, threshold, path):
            # TODO [I05]: h(node) pada f = g+h.
            f = g + ________("I05")
            if f > threshold:
                return f, None
            if node == goal:
                return -1, path + [node]
                
            min_threshold = float('inf')
            for neighbor in get_neighbors(grid, node[0], node[1]):
                if neighbor not in path:
                    temp, result_path = search(neighbor, g + 1, threshold, path + [node])
                    if temp == -1: 
                        return -1, result_path
                    if temp < min_threshold: 
                        min_threshold = temp
            return min_threshold, None

        threshold = manhattan(start, goal)
        while True:
            temp, result_path = search(start, 0, threshold, [])
            if temp == -1: 
                return result_path
            if temp == float('inf'): 
                return None
            threshold = temp
except NotImplementedError as error:
    print("BELUM DIISI:", error)

In [ ]:
# PENGUJIAN — jalankan setelah fungsi latihan diisi; tidak perlu diubah.
try:
    grid_1 = [
        [0, 0, 1, 0, 0],
        [0, 1, 0, 0, 1],
        [0, 0, 0, 1, 0],
        [1, 1, 0, 0, 0],
        [0, 0, 0, 1, 0]
    ]
    path_ida = idastar_grid(grid_1, (0,0), (4,4))
    assert path_ida is not None
    print('LULUS IDA* grid:', path_ida)
except NotImplementedError as error:
    print("BELUM DIISI:", error)

## 4. Recursive Best-First Search (RBFS)

RBFS menelusuri kandidat dengan nilai f terkecil secara rekursif. Pada setiap
tingkat, algoritma menyimpan batas dari alternatif terbaik. Jika kandidat utama
melampaui batas itu, pencarian kembali untuk mencoba alternatif.

Biaya anak dihitung dari $g(anak)=g(induk)+c(induk,anak)$, lalu
$f(anak)=\max(f(induk),g(anak)+h(anak))$. Setelah suatu cabang dikembalikan,
nilai f terbaiknya disimpan untuk menentukan pilihan berikutnya.

Memori penelusuran RBFS sekitar $O(bd)$. Dengan heuristik admissible dan biaya
langkah positif yang dibatasi di bawah, RBFS dapat menemukan solusi optimal,
tetapi pergantian cabang dapat menyebabkan banyak ekspansi ulang.

### Contoh: batas dari lintasan alternatif

In [ ]:
def rbfs_search(graph, heuristics, start, goal, stats=None):
    if stats is not None:
        stats.update(expanded=0, stack_peak=1, trace=[])
    def rbfs(node, g, node_f, f_limit, path):
        if stats is not None:
            stats['stack_peak'] = max(stats['stack_peak'], len(path) + 1)
        if node == goal:
            return True, path + [node], node_f
        if stats is not None:
            stats['expanded'] += 1
            stats['trace'].append(node)
            
        successors = graph.get(node, [])
        if not successors:
            return False, [], float('inf')
            
        succ_nodes = []
        for neighbor, cost in successors:
            if neighbor in path:
                continue
            total_g = g + cost
            child_f = max(total_g + heuristics[neighbor], node_f)
            succ_nodes.append([child_f, neighbor, cost])
        if not succ_nodes:
            return False, [], float('inf')
            
        while True:
            succ_nodes.sort(key=lambda x: x[0])
            best_f, best_node, best_cost = succ_nodes[0]
            
            if best_f == float('inf') or best_f > f_limit:
                return False, [], best_f
                
            alt_f = succ_nodes[1][0] if len(succ_nodes) > 1 else float('inf')
            
            result_bool, result_path, new_best_f = rbfs(
                best_node, g + best_cost, best_f, min(f_limit, alt_f), path + [node]
            )
            
            if result_bool:
                return True, result_path, best_f
                
            succ_nodes[0][0] = new_best_f

    success, path, _ = rbfs(start, 0, heuristics[start], float('inf'), [])
    return path if success else None

### Penelusuran RBFS pada contoh

Dari N1, kandidat N2 memiliki $g=3$, $h=4$, sehingga $f=7$.
Kandidat N3 memiliki $g=2$, $h=3$, sehingga $f=5$. RBFS memilih N3 dengan
batas alternatif 7.

Melalui N3, N4 memiliki $g=3$ dan $f=5$, sedangkan N5 memiliki $g=8$ dan $f=8$.
RBFS memilih N4, lalu mencapai N5 dengan biaya total 5.
Perhatikan bahwa g selalu menjumlahkan biaya sisi dari awal lintasan.

In [ ]:
graph_rbfs = {
    'N1': [('N2', 3), ('N3', 2)],
    'N2': [('N4', 4)],
    'N3': [('N4', 1), ('N5', 6)],
    'N4': [('N5', 2)],
    'N5': []
}

heuristics_rbfs = {
    'N1': 5, 'N2': 4, 'N3': 3, 'N4': 2, 'N5': 0
}

print("Lintasan RBFS:", rbfs_search(graph_rbfs, heuristics_rbfs, 'N1', 'N5'))

### Latihan dasar: RBFS pada grid

Lengkapi `rbfs_grid` untuk mencari lintasan dari `(4, 4)` ke `(0, 5)` pada
`grid_2`. Setiap perpindahan berbiaya 1. Gunakan `manhattan` dan `get_neighbors`
yang telah didefinisikan pada latihan IDA*.

Pertanyaan:
1. Mengapa g harus menjumlahkan biaya sisi, bukan sekadar menghitung panjang lintasan?
2. Apa peran nilai f alternatif dalam menentukan batas rekursi?
3. Kapan suatu cabang harus dinyatakan tidak memiliki solusi?

In [ ]:
# BAGIAN TUGAS — isi 1 penanda: I06.
try:
    def rbfs_grid(grid, start, goal):
        def rbfs(node, node_f, f_limit, path, g):
            if node == goal: 
                return True, path + [node], node_f
            
            neighbors = get_neighbors(grid, node[0], node[1])
            if not neighbors: 
                return False, [], float('inf')
            
            succ = []
            for nbr in neighbors:
                if nbr not in path:
                    # TODO [I06]: Akumulasi g dan h tetangga.
                    child_f = max(________("I06"), node_f)
                    succ.append([child_f, nbr, 1])
                    
            if not succ: 
                return False, [], float('inf')
            
            while True:
                succ.sort(key=lambda x: x[0])
                best_f, best_node, best_cost = succ[0]
                
                if best_f == float('inf') or best_f > f_limit: 
                    return False, [], best_f
                    
                alt_f = succ[1][0] if len(succ) > 1 else float('inf')
                res, res_path, best_f_new = rbfs(best_node, best_f, min(f_limit, alt_f), path + [node], g + best_cost)
                
                if res: 
                    return True, res_path, best_f
                succ[0][0] = best_f_new

        res, path, _ = rbfs(start, manhattan(start, goal), float('inf'), [], 0)
        return path if res else None
except NotImplementedError as error:
    print("BELUM DIISI:", error)

In [ ]:
# PENGUJIAN — jalankan setelah fungsi latihan diisi; tidak perlu diubah.
try:
    grid_2 = [
        [0, 0, 0, 1, 0, 0],
        [0, 1, 0, 0, 0, 0],
        [0, 1, 1, 1, 0, 1],
        [0, 0, 0, 1, 0, 0],
        [1, 1, 0, 0, 0, 0],
        [0, 0, 0, 1, 1, 0]
    ]
    path_rbfs = rbfs_grid(grid_2, (4,4), (0,5))
    assert path_rbfs is not None
    print('LULUS RBFS grid:', path_rbfs)
except NotImplementedError as error:
    print("BELUM DIISI:", error)

## 5. Pencarian dengan frontier terbatas

SMA* membatasi penggunaan memori dengan membuang kandidat terburuk, menyimpan
nilai f-nya pada induk, dan membangkitkan kembali kandidat tersebut jika diperlukan.

Contoh `sma_star_search` di bawah hanya memperagakan pemangkasan antrean.
Fungsi ini tidak menyimpan kembali nilai kandidat pada induk atau membangkitkannya
ulang, sehingga **bukan implementasi SMA* lengkap** dan tidak menjamin kelengkapan
maupun optimalitas. Batas `max_memory` menghitung entri antrean yang dipertahankan,
bukan seluruh penggunaan memori program.

### Contoh: memangkas antrean prioritas

In [ ]:
def sma_star_search(graph, heuristics, start, goal, max_memory=5, stats=None):
    """Demo frontier terbatas, BUKAN SMA* penuh: tanpa backup/regenerasi.

    max_memory membatasi jumlah entri frontier, bukan seluruh memori program.
    Tidak menjamin complete atau optimal.
    """
    if max_memory < 1:
        raise ValueError('max_memory minimal 1')
    if stats is not None:
        stats.update(pruned=0, frontier_peak=1, expanded=0, trace=[])
    open_list = [(heuristics[start], 0, [start])]
    
    while open_list:
        open_list.sort(key=lambda x: (x[0], -len(x[2])))
        current_f, current_g, path = open_list.pop(0)
        current_node = path[-1]
        
        if current_node == goal:
            return path, current_g
        if stats is not None:
            stats['expanded'] += 1
            stats['trace'].append(current_node)
            
        successors = graph.get(current_node, [])
        for neighbor, cost in successors:
            if neighbor not in path:
                g_new = current_g + cost
                f_new = max(current_f, g_new + heuristics[neighbor])
                open_list.append((f_new, g_new, path + [neighbor]))
                
        while len(open_list) > max_memory:
            open_list.sort(key=lambda x: (x[0], -len(x[2])))
            open_list.pop() 
            if stats is not None:
                stats['pruned'] += 1
        if stats is not None:
            stats['frontier_peak'] = max(stats['frontier_peak'], len(open_list))
            
    return None, float('inf')

Pada awal pencarian, R1 memiliki $g=0$ dan $f=4$. Setelah R1 diekspansi:

| Kandidat | g | h | f |
|---|---|---|---|
| R2 | 1 | 3 | 4 |
| R3 | 2 | 3 | 5 |

R2 diproses lebih dulu. Jika antrean melebihi kapasitas, kandidat dengan nilai f
terbesar dibuang sampai batas terpenuhi.

In [ ]:
graph_sma = {
    'R1': [('R2', 1), ('R3', 2)],
    'R2': [('R4', 3)],
    'R3': [('R4', 1), ('R5', 2)],
    'R4': [('R6', 2)],
    'R5': [('R6', 3)],
    'R6': []
}

heuristics_sma = {
    'R1': 4, 'R2': 3, 'R3': 3, 'R4': 2, 'R5': 2, 'R6': 0
}

sma_stats = {}
print('Versi sederhana:', sma_star_search(graph_sma, heuristics_sma, 'R1', 'R6', max_memory=2, stats=sma_stats))
assert sma_stats['pruned'] > 0 and sma_stats['frontier_peak'] <= 2
print('Pemangkasan:', sma_stats)

### Latihan dasar: membatasi ukuran frontier

Lengkapi `sma_star_grid` untuk mencari lintasan dari `(0, 0)` ke `(4, 4)` pada
`grid_3`. Batasi frontier menjadi paling banyak **3 entri** setelah pemangkasan.
Gunakan `manhattan` dan `get_neighbors` dari latihan IDA*.

Pertanyaan:
1. Kandidat mana yang dibuang ketika antrean penuh?
2. Mengapa solusi dapat hilang jika kandidat yang dibuang tidak dibangkitkan lagi?
3. Mengapa batas tiga entri bukan berarti seluruh program hanya menyimpan tiga simpul?

In [ ]:
# BAGIAN TUGAS — isi 1 penanda: I07.
try:
    def sma_star_grid(grid, start, goal, max_mem=3):
        open_list = [(manhattan(start, goal), 0, [start])]
        
        while open_list:
            open_list.sort(key=lambda x: (x[0], -len(x[2])))
            curr_f, curr_g, path = open_list.pop(0)
            curr_node = path[-1]
            
            if curr_node == goal: 
                return path
            
            for nbr in get_neighbors(grid, curr_node[0], curr_node[1]):
                if nbr not in path:
                    g_new = curr_g + 1
                    # TODO [I07]: Nilai f anak tidak menurun.
                    f_new = ________("I07")
                    open_list.append((f_new, g_new, path + [nbr]))
                    
            while len(open_list) > max_mem:
                open_list.sort(key=lambda x: (x[0], -len(x[2])))
                open_list.pop()
                
        return None
except NotImplementedError as error:
    print("BELUM DIISI:", error)

In [ ]:
# PENGUJIAN — jalankan setelah fungsi latihan diisi; tidak perlu diubah.
try:
    grid_3 = [
        [0, 0, 0, 0, 0],
        [0, 1, 1, 1, 0],
        [0, 0, 0, 1, 0],
        [1, 1, 0, 0, 0],
        [0, 0, 0, 1, 0]
    ]
    path_sma = sma_star_grid(grid_3, (0,0), (4,4), max_mem=3)
    assert path_sma is not None
    print('LULUS versi frontier terbatas:', path_sma)
except NotImplementedError as error:
    print("BELUM DIISI:", error)

## 6. Perbandingan pada labirin yang sama

Jalankan UCS, GBFS, dan A* pada `MAZE` dengan biaya langkah 1. Ketiganya memakai
urutan tetangga yang sama. Jika prioritas sama, kandidat yang lebih dulu masuk
diproses lebih dulu.

`expanded` menghitung simpul yang tetangganya dibangkitkan, tidak termasuk tujuan
atau entri antrean yang sudah usang. `frontier_peak` adalah jumlah entri antrean
terbanyak, bukan total memori program. Bandingkan biaya, jumlah ekspansi, dan waktu.
Pada animasi, biru menandai ekspansi dan hijau menandai lintasan hasil.

Percobaan berikutnya menggunakan labirin yang sama untuk IDA*, RBFS, dan pencarian
dengan frontier terbatas. `stack_peak` menunjukkan kedalaman rekursi maksimum.

Sel berikut mendefinisikan `best_first` dan `maze_search` beserta penghitungnya. Fungsi tersebut digunakan untuk membandingkan algoritma dengan aturan antrean dan pencatatan yang sama.

In [ ]:
def search_result(path, cost, expanded, frontier_peak, trace):
    return dict(path=path, cost=cost, expanded=expanded,
                frontier_peak=frontier_peak, trace=trace)


def grid_neighbors(grid, node):
    r, c = node
    for dr, dc in DIRECTIONS:
        nr, nc = r + dr, c + dc
        if 0 <= nr < len(grid) and 0 <= nc < len(grid[0]) and grid[nr][nc] == 0:
            yield (nr, nc), 1


def best_first(start, goal, neighbors, heuristic=lambda _: 0, weight=1, greedy=False):
    """UCS/A*/Weighted A*/GBFS; g-score membuka ulang state jika g membaik.

    expanded = state non-goal yang successor-nya dibangkitkan, termasuk re-expansion.
    Tie: FIFO; entri usang tidak dihitung. frontier_peak termasuk entri usang.
    """
    if weight < 0:
        raise ValueError('weight harus nonnegatif')
    serial = count()
    frontier = [(weight * heuristic(start), next(serial), 0, start)]
    costs, parents = {start: 0}, {start: None}
    trace, peak = [], 1
    while frontier:
        _, _, g, node = heapq.heappop(frontier)
        if g != costs[node]:
            continue
        if node == goal:
            path = []
            while node is not None:
                path.append(node)
                node = parents[node]
            return search_result(path[::-1], g, len(trace), peak, trace)
        trace.append(node)
        for child, step_cost in neighbors(node):
            if step_cost <= 0:
                raise ValueError('Eksperimen ini mensyaratkan biaya edge positif')
            new_g = g + step_cost
            if new_g < costs.get(child, float('inf')):
                costs[child], parents[child] = new_g, node
                priority = heuristic(child) if greedy else new_g + weight * heuristic(child)
                heapq.heappush(frontier, (priority, next(serial), new_g, child))
        peak = max(peak, len(frontier))
    return search_result(None, float('inf'), len(trace), peak, trace)


def maze_search(method, grid=MAZE, start=START, goal=GOAL):
    if method not in ('UCS', 'GBFS', 'A*'):
        raise ValueError(method)
    neighbors = lambda node: grid_neighbors(grid, node)
    h = (lambda node: manhattan(node, goal)) if method != 'UCS' else (lambda _: 0)
    return best_first(start, goal, neighbors, h, greedy=method == 'GBFS')

In [ ]:
maze_results = {}
print('Metode | Langkah | Ekspansi | Maks. frontier/tumpukan | Detik')
for method in ('UCS', 'GBFS', 'A*'):
    before = perf_counter()
    row = maze_search(method)
    maze_results[method] = row
    print(method, row['cost'], row['expanded'], row['frontier_peak'], f"{perf_counter()-before:.6f}", sep=' | ')
assert maze_results['A*']['cost'] == 8
print('LULUS perbandingan maze')
display(animate_expansion(MAZE, maze_results['A*']))

In [ ]:
maze_graph = {(r,c):list(grid_neighbors(MAZE, (r,c))) for r,row in enumerate(MAZE) for c,value in enumerate(row) if value == 0}
maze_h = {node:manhattan(node, GOAL) for node in maze_graph}
ida_maze_stats, rbfs_maze_stats, limited_maze_stats = {}, {}, {}
# h=0 pada IDA* sengaja dipakai untuk memperlihatkan kenaikan threshold.
ida_maze_path, _ = idastar_search(maze_graph, dict.fromkeys(maze_graph, 0), START, GOAL, ida_maze_stats)
rbfs_maze_path = rbfs_search(maze_graph, maze_h, START, GOAL, rbfs_maze_stats)
limited_maze_path, limited_maze_cost = sma_star_search(maze_graph, maze_h, START, GOAL, max_memory=3, stats=limited_maze_stats)
for label, path, stats in [('IDA* h=0', ida_maze_path, ida_maze_stats), ('RBFS Manhattan', rbfs_maze_path, rbfs_maze_stats), ('Frontier terbatas M=3', limited_maze_path, limited_maze_stats)]:
    assert len(path)-1 == 8
    summary = {k:v for k,v in stats.items() if k != 'trace'}
    print(label, 'langkah', len(path)-1, summary)
assert len(ida_maze_stats['thresholds']) > 1
print('LULUS perbandingan pada labirin yang sama')

## 7. Perbandingan heuristik pada 8-puzzle

Gunakan keadaan awal `(7,2,4,5,0,6,8,3,1)` dan tujuan `(1,2,3,4,5,6,7,8,0)`.
Jarak optimalnya 20 langkah. Bandingkan tiga heuristik: nilai nol, jumlah ubin
yang salah posisi (*misplaced tiles*), dan jarak Manhattan. Sel kosong 0 tidak dihitung.

Ketiga percobaan memakai implementasi A*, urutan tetangga, dan aturan penghitung
yang sama. Dengan pengaturan ini, jumlah ekspansinya berturut-turut 51.759,
3.666, dan 282, dengan biaya solusi tetap 20. Jelaskan mengapa heuristik dapat
mengurangi ekspansi tanpa mengubah biaya solusi.

Fungsi tetangga dan ketiga heuristik untuk tuple datar didefinisikan pada sel berikut. Latihan puzzle sebelumnya menggunakan tuple tiga baris.

In [ ]:
def puzzle_neighbors(state):
    blank = state.index(0)
    r, c = divmod(blank, 3)
    for dr, dc in DIRECTIONS:
        nr, nc = r + dr, c + dc
        if 0 <= nr < 3 and 0 <= nc < 3:
            child = list(state)
            j = 3 * nr + nc
            child[blank], child[j] = child[j], child[blank]
            yield tuple(child), 1


def misplaced(state):
    return sum(tile != 0 and tile != PUZZLE_GOAL[i] for i, tile in enumerate(state))


def puzzle_manhattan(state):
    return sum(manhattan(divmod(i, 3), divmod(tile - 1, 3))
               for i, tile in enumerate(state) if tile)

In [ ]:
puzzle_results = {}
print('Heuristik | Langkah | Ekspansi | Maks. frontier | Detik')
for label, h_fn in [('h=0', lambda _: 0), ('misplaced', misplaced), ('Manhattan', puzzle_manhattan)]:
    before = perf_counter()
    row = best_first(PUZZLE_START, PUZZLE_GOAL, puzzle_neighbors, h_fn)
    puzzle_results[label] = row
    assert row['cost'] == 20
    print(label, row['cost'], row['expanded'], row['frontier_peak'], f'{perf_counter()-before:.4f}', sep=' | ')
assert [puzzle_results[k]['expanded'] for k in ('h=0', 'misplaced', 'Manhattan')] == [51759, 3666, 282]
assert misplaced(PUZZLE_GOAL) == puzzle_manhattan(PUZZLE_GOAL) == 0
print('LULUS perbandingan heuristik')

### Latihan lanjutan: mengubah heuristik

1. Ganti Manhattan dengan `max(misplaced, Manhattan)`, lalu dengan `2*Manhattan`.
   Catat biaya solusi, jumlah ekspansi, dan waktu.
2. Periksa konsistensi pada sisi-sisi dari 100 keadaan pertama yang diekspansi
   UCS. Pemeriksaan sebagian dapat menemukan pelanggaran, tetapi belum membuktikan
   konsistensi pada seluruh ruang keadaan.
3. Jelaskan mengapa penjumlahan dua heuristik admissible belum tentu admissible.

### Tugas mandiri: Weighted A*

Tulis `weighted_a_star` dengan prioritas $g+w h$. Simpan biaya terbaik,
abaikan entri yang usang, dan proses kembali keadaan jika ditemukan biaya lebih
kecil. `w=1` harus berperilaku seperti A* dan `w=0` seperti UCS.

Bandingkan `w=1, 1.5, 2` pada puzzle yang sama. Untuk `w>1`, solusi tidak selalu
optimal. Sertakan biaya dan jumlah ekspansi dalam hasil percobaan.

In [ ]:
# BAGIAN TUGAS — isi 1 penanda: I08.
try:
    def weighted_a_star(start, goal, neighbors, h_fn, w=1):
        # TODO [I08]: Tulis seluruh isi weighted_a_star sesuai petunjuk Weighted A*.
        # __ MULAI JAWABAN __
        ________("I08")
        # __ AKHIR JAWABAN __

    # PENGUJIAN — jalankan setelah fungsi di atas diisi; tidak perlu diubah.
    for w in (0, 1, 1.5, 2):
        row = weighted_a_star(PUZZLE_START, PUZZLE_GOAL, puzzle_neighbors, puzzle_manhattan, w)
        assert row['path'][0] == PUZZLE_START and row['path'][-1] == PUZZLE_GOAL
        assert all(any(child == b for child, _ in puzzle_neighbors(a)) for a, b in zip(row['path'], row['path'][1:]))
        if w in (0, 1):
            assert row['cost'] == 20
        print('Weighted A*', w, 'biaya', row['cost'], 'ekspansi', row['expanded'])
    print('LULUS Weighted A*')
except NotImplementedError as error:
    print("BELUM DIISI:", error)

## 8. Ringkasan

| Metode | Prioritas | Kelengkapan dan optimalitas | Waktu dan memori |
|---|---|---|---|
| GBFS | $h$ | Tidak dijamin secara umum | Kasus terburuk pohon pencarian: waktu dan memori $O(b^m)$ |
| A* | $g+h$ | Lengkap dan optimal dengan syarat pada bagian A* | Waktu dan memori dapat eksponensial |
| IDA* | DFS dengan batas f | Lengkap dan optimal dengan heuristik admissible dan syarat biaya positif | Banyak ekspansi ulang; memori penelusuran $O(bd)$ |
| RBFS | f dan batas alternatif | Lengkap dan optimal dengan heuristik admissible dan syarat biaya positif | Dapat menelusuri ulang cabang; memori $O(bd)$ |
| Frontier terbatas pada contoh | f, lalu buang kandidat terburuk | Tidak dijamin | Paling banyak M entri dipertahankan; penyimpanan lintasan tetap memerlukan memori |

Pada graf berhingga dengan pencatatan kunjungan, GBFS dapat menyelesaikan
pencarian, tetapi belum tentu menghasilkan biaya minimum. Pembatasan antrean
saja tidak memberikan jaminan SMA* lengkap.

## Referensi

- Russell, S. & Norvig, P. *Artificial Intelligence: A Modern Approach*, edisi ke-4, Bab 3.
  [Daftar isi](https://aima.cs.berkeley.edu/contents.html).
- [Kode pendamping AIMA](https://github.com/aimacode/aima-python).